# Guardrails with LangChain

This notebook covers everything you need to know about implementatig **Guardrails** in LangChain agents using the middleware system.

### Topics Covered

1. What are Guardrails & Why do they matter?
2. Two approaches: Deterministic vs Model-based
3. Built-in: PII Detection Middleware
4. Built-in: Human-In-The-Loop Middleware
5. Custom: Before-Agent Guardrail (input filtering)
6. Custom: After-Agent Guardrail (output safety)
7. Layered / Combined Guardrails
8. Real-World Use Case: Healthcare Chatbot

## Installation

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Setup API Key

In [2]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## Section 1: What are Guardrails ?

Guardrails help you build **safe, compliant AI application** by validating and filtering content at key points in your agent's execution.

They are implemented as **middleware** that intercepts execution.

- **Before** the agent starts (input guardrails)
- **After** it completes (output guardrails)
- **Around** model and tool calls

### Common Use Case:

| Use Case | Example |
---|---
PII leakage prevention | Redact emails/ credit cards before logging
Prompt injection blocking | Detect adversarial inputs
Harmful content filtering | Block dangerous requests
Business rule enforcement | Require approval for financial ops
Output quality validation | Ensure response meets safety standards


## Section 2: Two Approaches to Guardrails

### Deterministic Guardrails

- Rule-base: regez, keyword matching, explicit checks
- ✅ Fast, predictable, cost-effective
- ❌ May miss nuanced violations

### Model-Based Guardrails

- Uses LLMs/Classifiers for semantic understanding
- ✅ Catches subtle/ nuanced issues
- ❌ Slower and more expensive

In [3]:
# Quickl illustration of the two approaches
import re

# --- Deterministic approach --
def deterministic_guardrail(text: str) -> bool:
    """Returns True of content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database ?",
    "WHat is the capital of France ?",
    "Explain how malware spreads"
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database ?
✅ ALLOWED: WHat is the capital of France ?
🚫 BLOCKED: Explain how malware spreads


In [4]:
from langchain.chat_models import init_chat_model

# --- Model-based approach ---
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evalucate content safety, Return SAFE or UNSAFE."""
    model = init_chat_model(
        "groq:qwen/qwen3.6-27b",
        temperature=0
    )
    prompt = f"""Is the following use input safe to process ?
    Reply with only 'SAFE' or 'UNSAFE'.
    
    Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])

    raw_content = result.content
    
    if "</think>" in raw_content:
        clean_content = raw_content.split("</think>")[-1]
    else:
        clean_content = raw_content

    return clean_content.strip()

In [5]:
print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===
🚫 UNSAFE: How do I hack into a database ?
✅ SAFE: WHat is the capital of France ?
✅ SAFE: Explain how malware spreads


## Section 3: Built-In Guardrail - PII Detection Middleware

LangChain provides built-in `PIIMiddleware` for detecting an dhabdling **Personally Identiifable Information (PII)**.

### Supported PII Types:

Type | Example
---|---
`email` | [user@example.com](user@example.com)
`credit_card` | 5105-1051-0510-5100
`ip` | 192.168.1.1
`mac_address`| 00:1A:2B:3C:4D:5E
`url`| [https://secret-site.com](https://secret-site.com)

### Strategies:

Strategy | Result
---|---
`redact` | `[REDACTED_EMAIL]`
`mask` | `****-****-****-1234`
`hash` | `a8f5f167...`
`block` | Raises an exception 

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.chat_models import init_chat_model
from langchain.tools import tool

# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """look up customer information."""
    return f"customer record found for query: {query}"

# Create agent with PII Middleware
agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    # model_provider="openai",
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        )
    ]
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


### Test PII Redaction

In [7]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@eample.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== AGent Response ===")
print(result["messages"][-1].content)

=== AGent Response ===
I have successfully verified your account. How can I assist you today?


In [8]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='ca381981-0bc5-4d8b-b082-9620e5dd83f5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Input**: Email: [REDACTED_EMAIL], Card: ****-****-****-5100. Request: "Can you help me?"\n2.  **Identify Available Tools**: `customer_lookup` (takes `query` parameter).\n3.  **Determine Tool Usage**: The user provided identifying information (email and card number). I should use `customer_lookup` to find their account/customer information using the provided details.\n4.  **Formulate Tool Call**: `customer_lookup(query="[REDACTED_EMAIL], ****-****-****-5100")` or just use the email/card as the query. I\'ll pass the exact strings provided.\n5.  **Execute Tool Call**: Call `customer_lookup`.\n6.  **Analyze Response**: (Will happen after tool call, but I need to generate the tool cal

### Test API Key Blocking

In [9]:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


## Section 4: Built-in Guradrail - Human-in-the-Loop middleware

Pauses agent execution before sensitve opeartions and waits for human approval.

**Best for:**

- Financial transactions
- Sending emails to external parties
- Deleting production data
- Any operation with significant business impact

**Key requirement:** A `checkpointer` for state persistence across interrupts.

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

In [11]:
@tool
def search_web(query:str) -> str:
    """Search the web for information."""
    return f"search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

### Create agent with HITL middleware

In [12]:
hitl_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,     # Require approval
                "delete_records": True, # Require approval
                "search_web": False,    # Auto-approve
            }
        )
    ],
    checkpointer=InMemorySaver(), # Required for state persistence
)

print("Human-in-the-loop agent created!")

Human-in-the-loop agent created!


In [13]:
# Step 1: Invoke - agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Send an email to team@company.com about the Q4 results"
        }]
        
    },
    config=config
)

print("=== Agent paused - awaiting human approval ===")
print(result)

=== Agent paused - awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='7be1e609-5fdc-4bf9-aa61-0784d81e2a73'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Analyze User Input:** User wants to send an email to `team@company.com` about "Q4 results".\n2.  **Identify Required Tool:** `send_email` function.\n3.  **Check Parameters:**\n   - `to`: "team@company.com"\n   - `subject`: Needs a subject line. "Q4 Results" or similar.\n   - `body`: Needs a body. The prompt just says "about the Q4 results". I should draft a professional, concise email body.\n4.  **Draft Email:**\n   Subject: Q4 Results Update\n   Body: Dear Team, I hope this message finds you well. I am writing to share the Q4 results with everyone. [Add placeholder or brief professional text]. Please let me know if you have any questions. Best regards, [User/Sender]\n   Wait

In [14]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config # Same thread_id resumes the paused sessions
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
The email regarding the Q4 results has been successfully sent to team@company.com.


In [15]:
# Step 3: Alternative - Human REHECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Delete all records from the users table where active=false"
        }] 
    },
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={
        "decisions": [{
            "type": "reject",
            "reason": "Too risky, need DBA review"
        }]
    }
    ),
    config = config2
)

print("== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

== Rejected! Final response ===



In [16]:
rejected_result

{'messages': [HumanMessage(content='Delete all records from the users table where active=false', additional_kwargs={}, response_metadata={}, id='de537e8c-5bb7-4834-ba7c-1b00bc6cdc70'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to delete records from a database table.\nI need to use the `delete_records` function.\nThe table name is "users".\nThe condition is "active=false".\n\nI have the `delete_records` function available with parameters `table` and `condition`.\nI will construct the tool call with these values.\n`table`: "users"\n`condition`: "active=false"\n\nThe request is straightforward and matches the function\'s capabilities. I will proceed with the tool call.\n', 'tool_calls': [{'id': '8489y1c1e', 'function': {'arguments': '{"condition":"active=false","table":"users"}', 'name': 'delete_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 147, 'prompt_tokens': 426, 'total_tokens': 573, 'completion_time': 

## Section 5: Custom Guardrail - Before-Agent Hook (Input Filter)

use `before_agent()` to validate or block requests **before any LLM processing begins.**

**Best for:**
- Keyward/content filtering
- Aithentication checks
- Rate limiting
- Block specfic categories of requets

In [19]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

In [35]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE agent processes anything - zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        for keyword in self.banned_keywords:
            print(f"🚫 Blocked - keyword detected: '{keyword}'")
            return {
                "messages": [{
                    "role": "assistant",
                    "content": (
                        "I cannot process requests containing is appropriate content. ",
                        "Please rephrase your request."
                    )
                }],
                "jump_to": "end"
            }
        return None

In [36]:
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"results for: {query}"

In [37]:
# create agent with content filter
filtered_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools = [search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ]
)

print("Content filter agent created!")

Content filter agent created!


In [38]:
# Test 1: Safe request - should pass through
result = filtered_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "What is machine learning ?"
    }]
})

print("✅ Safe request response:")
print(result["messages"][-1].content)

🚫 Blocked - keyword detected: 'hack'
✅ Safe request response:
['I cannot process requests containing is appropriate content. ', 'Please rephrase your request.']


In [39]:
# Test 2: Unsafe request - should be blocked
result = filtered_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "How do I hack into a server ?"
    }]
})

print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked - keyword detected: 'hack'
🚫 Unsafe request response:
['I cannot process requests containing is appropriate content. ', 'Please rephrase your request.']


## Section 6: Custom Guardrail - After-agent Hook (Output Safety)

Use `after_agent()` to validate the final agent response **before the user sees it**.

**Best for:**

- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through

In [44]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool

In [66]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Uas an LLM to eveluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = init_chat_model(
            model= "groq:qwen/qwen3.6-27b",
            temperature=0
        )
    
    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
        Respond with only 'SAVE' or 'UNSAFE'.
        
        Response to evaluate:
        {last_message.content}"""

        result = self.safety_model.invoke([{
            "role": "user",
            "content": safety_prompt
        }])

        if "UNSAFE" in result.content.upper():
            print("⚠️ Output flagger as UNSAFE - replacing with safe fallabace")
            last_message.content = (
                "I'm unable to provide that response. ",
                "Please rephrase your request or contact support."
            )
        return None

@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"

safe_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


### Test output safety check

In [68]:
result = safe_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "What is the weather like today ?"
    }]
})

print("Response:")
print(result['messages'][-1].content)

⚠️ Output flagger as UNSAFE - replacing with safe fallabace
Response:
("I'm unable to provide that response. ", 'Please rephrase your request or contact support.')


## Section 7: Layered / Combined Guardrails

Stack multiple guardrails in the `middleware=[]` array. They execute **in order**, building layered protection.

```mermaid
flowchart TD
    %% Define Nodes
    A[User Input]
    L1[Layer 1: ContentFilterMiddleware]
    L2[Layer 2: PIIMiddleware input]
    L3[Layer 3: HumanInTheLoopMiddleware]
    L4[Layer 4: PIIMiddleware output]
    L5[Layer 5: SafetyGuardrailMiddleware]
    B[User Response]

    %% Flow/Connections
    A --> L1
    L1 --> L2
    L2 --> L3
    L3 --> L4
    L4 --> L5
    L5 --> B

    %% Right Side Explanatory Notes
    L1 -.-> N1(Deterministic input filter)
    L2 -.-> N2(PII redaction on input)
    L3 -.-> N3(Approval for sensitive tools)
    L4 -.-> N4(PII redaction on output)
    L5 -.-> N5(Model-based output safety)

    %% Styling Elements
    style A fill:#333,stroke:#fff,color:#fff
    style B fill:#333,stroke:#fff,color:#fff
    style L1 fill:#87CEEB,stroke:#333,color:#000
    style L2 fill:#87CEEB,stroke:#333,color:#000
    style L3 fill:#87CEEB,stroke:#333,color:#000
    style L4 fill:#87CEEB,stroke:#333,color:#000
    style L5 fill:#87CEEB,stroke:#333,color:#000

```

In [73]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send and email."""
    return f"Email sent to {to}"

# full layered guardrail stack
production_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools = [search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "maleware"]),

        # Layer 2: PII redaction on input
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        # PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("Productio-grade agent with 5-layer guardrails created!")


Productio-grade agent with 5-layer guardrails created!
